- 데이터 정제 후 axencoder-len512에 batch=128.lr=4.8e-4 적용 풀런

In [1]:
import sys
sys.path.insert(0, "src")  # /workspace/src

from patent_train import TrainingRunner, TrainConfig, probe_batches

In [2]:
SEARCH = False   # True=fast-fail 짧은 런(2 epoch로 유도) / False=풀런(아래 epochs 사용)

cfg = TrainConfig(
    backbone="axenc",   # backbones.BACKBONES 키
    loss="focal",
    loss_params={"alpha": 0.25, "gamma": 2},
    max_len=512,
    eff_batch=128,          # 배치 재현 파라미터
    micro_batch=128,        
    eval_micro_batch=512,
    learning_rate=4.8e-4,   # 확정 레시피 lr
    weight_decay=0.01,
    warmup_ratio=0.1,
    epochs=12,
    early_stop_epochs=2,    # 개선 없이 견디는 에폭 수(eval 횟수 환산은 runner가 처리)
    notebook_name="11_01_CleanData_Recipe.ipynb",   # wandb code saving
    tag="modernbert-patent-len512-b128",
    run_name="axenc_len512_focal_eff128_lr4.8e4_clean",
    repo_final="ingyoun/A.X-patent-len512-b128",
    out_path="/workspace/output/modernbert-len512-b128",
    search=SEARCH,
)

print("run_name:", cfg.run_name, "| epochs:", cfg.epochs, "| grad_accum:", cfg.grad_accum)

run_name: axenc_len512_focal_eff128_lr4.8e4_clean | epochs: 12 | grad_accum: 1


## 구성 — 데이터·모델

토크나이저+원본 로드 → `max_len` 절단(캐시) → 분류기 구성. 단계를 나눠 중간 점검·부분 재실행이 가능하다.

In [3]:
runner = TrainingRunner(cfg)

In [4]:
runner.load_data()        # 토크나이저 + 원본 데이터셋(prep 캐시 있으면 원본 생략)
runner.data.raw

README.md:   0%|          | 0.00/679 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/531M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/519M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/542M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/89.6M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/87.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/201616 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11244 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/11132 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 201616
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11244
    })
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11132
    })
})

In [5]:
runner.prepare_data()     # max_len 절단 → prep 캐시
runner.data.dataset

Map:   0%|          | 0/201616 [00:00<?, ? examples/s]

Map:   0%|          | 0/11244 [00:00<?, ? examples/s]

Map:   0%|          | 0/11132 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/201616 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/11244 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/11132 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 201616
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 11244
    })
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 11132
    })
})

In [6]:
runner.load_model()

[transformers] Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertForSequenceClassification is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B", attn_implementation="flash_attention_2", dtype=torch.float16)`
[transformers] Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertModel is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B", attn_implementation="flash_attention_2", dtype=torch.float16)`


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: skt/A.X-Encoder-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## OOM 확인

GPU/배치에서 안전한 `micro_batch` 상한을 실측

In [7]:
# probe_batches(
#     runner.model, runner.data.tokenizer.vocab_size, cfg.max_len,
#     train_mb=(32, 64, 96, 128, 160),
#     eval_mb=(128, 256, 512),
# )

## 훈련

In [8]:
runner.build_trainer()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


[schedule] 1576 step/epoch | eval·save 788 step마다(2회/epoch) | early stop 2 epoch(patience=4 eval)


In [9]:
runner.train()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: paraise (paraise-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Micro F1,Macro F1,Sample F1,Empty Rate,Anchor Weighted F1
788,0.001437,0.000907,0.638039,0.591482,0.567105,0.286202,0.651989
1576,0.000877,0.000664,0.719535,0.689049,0.675355,0.190262,0.724893
2364,0.000642,0.000541,0.763810,0.747187,0.738227,0.126752,0.753496
3152,0.000555,0.000500,0.790542,0.779267,0.776187,0.090370,0.763753
3940,0.000497,0.000478,0.791781,0.779451,0.778482,0.091897,0.774831
4728,0.000503,0.000481,0.795084,0.781964,0.785734,0.081567,0.779868
5516,0.000466,0.000468,0.810149,0.799947,0.810322,0.056414,0.784803
6304,0.000466,0.000451,0.805735,0.795740,0.792409,0.086867,0.788035
7092,0.000389,0.000450,0.820618,0.812669,0.822045,0.047610,0.789637
7880,0.000404,0.000463,0.808259,0.795060,0.807900,0.062792,0.788553


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 평가 · 메트릭 저장 · push

모델 가중치는 로컬에 두지 않고 Hub로만 올린다(팟을 지우면 로컬 사본은 사라진다). 로컬 사본이 필요하면 `runner.save_model()`.

In [10]:
test_metrics = runner.evaluate("test")   # 05_01 원본(0.8600) 대비 회귀 확인
for k, v in test_metrics.items():
    print(f"{k}: {v}")

[transformers] early stopping required metric_for_best_model, but did not find eval_micro_f1 so early stopping is disabled


Training Loss,Validation Loss,Step,Micro F1,Macro F1,Sample F1,Empty Rate,Anchor Weighted F1
0.000021,0.000852,18912,0.858759,0.856503,0.873791,0.011740,0.821451


test_loss: 0.0008520814590156078
test_micro_f1: 0.8587587219977966
test_macro_f1: 0.8565032351742721
test_sample_f1: 0.8737905262451687
test_empty_rate: 0.011739594450373533
test_anchor_weighted_f1: 0.8214510389738382


In [11]:
runner.save_metrics()     # runner.metrics(split 전체) → {tag}_metrics.json

[save] /workspace/output/modernbert-len512-b128/modernbert-patent-len512-b128_metrics.json  splits=['test']


In [12]:
runner.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

[push] ingyoun/A.X-patent-len512-b128


## val·test 로짓 덤프

`logits_{tag}_{split}.npy`를 `out_path` 상위(`/workspace/output/`)에 저장

In [13]:
runner.predict_logits("val")
runner.predict_logits("test")

[dump] /workspace/output/logits_modernbert-patent-len512-b128_val.npy  shape=(11132, 188)


[dump] /workspace/output/logits_modernbert-patent-len512-b128_test.npy  shape=(11244, 188)


array([[-10.4375 ,  -6.25   ,  -8.625  , ..., -10.     ,  -8.0625 ,
         -8.125  ],
       [ -8.1875 ,  -7.40625,  -7.21875, ...,  -6.78125,  -5.96875,
         -7.84375],
       [-10.125  ,  -8.9375 ,  -7.125  , ...,  -9.375  ,  -8.5625 ,
         -7.96875],
       ...,
       [ -5.65625,  -6.5    ,  -6.15625, ...,  -6.21875,  -5.28125,
         -6.25   ],
       [ -8.125  ,  -7.25   , -10.1875 , ...,  -8.0625 ,  -8.125  ,
         -7.8125 ],
       [ -7.53125,  -8.1875 ,  -6.96875, ...,  -7.125  ,  -7.15625,
         -7.53125]], shape=(11244, 188), dtype=float32)